# Strokes Annotation Loader Test
Loads every `_strokes.json` file in the project folder, validates the data, reconstructs binary masks, and visualises the results.

## 1 · Import libraries

In [4]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

## 2 · Load strokes JSON files

In [7]:
ANNO_DIR = Path(".")  # folder containing _strokes.json files

stroke_files = sorted(ANNO_DIR.glob("*_strokes.json"))
print(f"Found {len(stroke_files)} strokes file(s):")
for p in stroke_files:
    print(" ", p.name)

# Load all cases into a list
cases = []
for p in stroke_files:
    data = json.loads(p.read_text())
    cases.append(data)
    H, W = data["image_size"]
    n_strokes = len(data["strokes"])
    n_pts = sum(s["n_points"] for s in data["strokes"])
    print(f"\n{p.name}")
    print(f"  image : {data['image']}  ({H}×{W})")
    print(f"  structures : {[s['name'] for s in data['structures']]}")
    print(f"  strokes : {n_strokes}  |  total points : {n_pts}")

Found 2 strokes file(s):
  00_000001-1.2.840.4267.32.204129199654584020110347304149310718377_strokes.json
  00_000001-1.2.840.4267.32.325406104635607657178581763291090652390_strokes.json

00_000001-1.2.840.4267.32.204129199654584020110347304149310718377_strokes.json
  image : 00_000001-1.2.840.4267.32.204129199654584020110347304149310718377.jpg  (812×840)
  structures : ['ETT', 'NG / OG tube', 'Central line (CVC)', 'Chest tube', 'PICC']
  strokes : 4  |  total points : 611

00_000001-1.2.840.4267.32.325406104635607657178581763291090652390_strokes.json
  image : 00_000001-1.2.840.4267.32.325406104635607657178581763291090652390.jpg  (840×812)
  structures : ['ETT', 'NG / OG tube', 'Central line (CVC)', 'Chest tube', 'PICC']
  strokes : 3  |  total points : 378


## 3 · Parse and validate structure

In [8]:
REQUIRED_TOP = {"image", "image_size", "structures", "strokes"}
REQUIRED_STROKE = {"stroke_index", "structure", "sid", "instance", "n_points", "points"}

errors = []

for case in cases:
    label = case["image"]
    missing_top = REQUIRED_TOP - case.keys()
    if missing_top:
        errors.append(f"{label}: missing top-level keys {missing_top}")
        continue

    H, W = case["image_size"]
    ch_map = {s["name"]: s["channel"] for s in case["structures"]}

    for i, stroke in enumerate(case["strokes"]):
        missing = REQUIRED_STROKE - stroke.keys()
        if missing:
            errors.append(f"{label} stroke {i}: missing keys {missing}")
            continue

        pts = np.array(stroke["points"])
        if pts.ndim != 2 or pts.shape[1] != 3:
            errors.append(f"{label} stroke {i}: points must be (N,3) — got {pts.shape}")
            continue

        # Check coordinates are within image bounds
        oob_x = np.sum((pts[:, 0] < 0) | (pts[:, 0] >= W))
        oob_y = np.sum((pts[:, 1] < 0) | (pts[:, 1] >= H))
        if oob_x or oob_y:
            errors.append(f"{label} stroke {i}: {oob_x} x and {oob_y} y coords out of bounds ({W}×{H})")

        # Check structure name is known
        if stroke["structure"] not in ch_map:
            errors.append(f"{label} stroke {i}: unknown structure '{stroke['structure']}'")

        # Check declared n_points matches actual array length
        if stroke["n_points"] != len(pts):
            errors.append(f"{label} stroke {i}: n_points={stroke['n_points']} but {len(pts)} points in array")

if errors:
    print(f"❌  {len(errors)} validation error(s):")
    for e in errors:
        print(" ", e)
else:
    total = sum(len(c["strokes"]) for c in cases)
    print(f"✅  All {len(cases)} case(s), {total} stroke(s) validated successfully.")

✅  All 2 case(s), 7 stroke(s) validated successfully.


## 4 · Reconstruct masks and visualise

Each strokes file is rendered to a `(H, W, N_structures)` uint8 mask using the saved `[x, y, width]` triplets, then shown alongside the source image (if present in the same folder).

In [ ]:
def load_strokes(data: dict) -> np.ndarray:
    """Return (H, W, N_structures) uint8 mask reconstructed from [x,y,width] stroke points."""
    H, W = data["image_size"]
    N = len(data["structures"])
    ch_map = {s["name"]: s["channel"] for s in data["structures"]}
    # Draw on separate contiguous 2D arrays; cv2 requires contiguous memory
    channels = [np.zeros((H, W), dtype=np.uint8) for _ in range(N)]
    for stroke in data["strokes"]:
        ch = ch_map.get(stroke["structure"], -1)
        if ch < 0:
            continue
        pts = np.array(stroke["points"])   # (K, 3): x, y, width
        for i in range(len(pts) - 1):
            x0, y0, w0 = pts[i]
            x1, y1, w1 = pts[i + 1]
            thickness = max(1, int(round((w0 + w1) / 2)))
            cv2.line(channels[ch],
                     (int(x0), int(y0)), (int(x1), int(y1)),
                     color=255, thickness=thickness, lineType=cv2.LINE_AA)
    return np.stack(channels, axis=-1)


def hex_to_rgb(h: str) -> tuple:
    return tuple(int(h.lstrip("#")[i:i+2], 16) / 255 for i in (0, 2, 4))


for case in cases:
    H, W = case["image_size"]
    structures = case["structures"]
    mask = load_strokes(case)

    # Build an RGB composite overlay: blend each structure channel with its colour
    overlay = np.zeros((H, W, 3), dtype=np.float32)
    for s in structures:
        ch = s["channel"]
        colour = np.array(hex_to_rgb(s["color"]), dtype=np.float32)
        alpha = (mask[:, :, ch] / 255.0)[:, :, np.newaxis]
        overlay = np.maximum(overlay, alpha * colour)

    # Try to load the source image for comparison
    img_path = ANNO_DIR / case["image"]
    source = np.array(Image.open(img_path).convert("RGB")) if img_path.exists() else None

    ncols = 3 if source is not None else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
    fig.suptitle(case["image"], fontsize=10)

    col = 0
    if source is not None:
        axes[col].imshow(source, cmap="gray")
        axes[col].set_title("Source image")
        axes[col].axis("off")
        col += 1

    axes[col].imshow(source if source is not None else np.zeros((H, W), dtype=np.uint8),
                     cmap="gray", alpha=0.6 if source is not None else 1.0)
    axes[col].imshow(overlay, alpha=0.8 if source is not None else 1.0)
    patches = [mpatches.Patch(color=hex_to_rgb(s["color"]), label=s["name"]) for s in structures
               if mask[:, :, s["channel"]].any()]
    axes[col].legend(handles=patches, loc="lower right", fontsize=8)
    axes[col].set_title("Reconstructed annotation overlay")
    axes[col].axis("off")
    col += 1

    axes[col].imshow(overlay)
    axes[col].set_title("Annotation only")
    axes[col].axis("off")

    plt.tight_layout()
    plt.show()

error: OpenCV(4.10.0) :-1: error: (-5:Bad argument) in function 'line'
> Overload resolution failed:
>  - Layout of the output array img is incompatible with cv::Mat
>  - Expected Ptr<cv::UMat> for argument 'img'


## 5 · Assertion tests

In [6]:
passed = 0
failed = 0

def check(condition: bool, msg: str):
    global passed, failed
    if condition:
        print(f"  ✅  {msg}")
        passed += 1
    else:
        print(f"  ❌  {msg}")
        failed += 1


for case in cases:
    print(f"\n── {case['image']} ──")
    H, W = case["image_size"]
    structures = case["structures"]
    strokes = case["strokes"]
    mask = load_strokes(case)

    check(len(strokes) > 0, f"At least one stroke present ({len(strokes)})")
    check(len(structures) > 0, f"At least one structure defined ({len(structures)})")
    check(mask.shape == (H, W, len(structures)),
          f"Mask shape {mask.shape} matches (H={H}, W={W}, N={len(structures)})")

    for s in structures:
        ch = s["channel"]
        annotated = mask[:, :, ch].any()
        # Only assert annotated if any stroke targets this structure
        has_strokes = any(st["structure"] == s["name"] for st in strokes)
        if has_strokes:
            check(annotated, f"Structure '{s['name']}' (ch {ch}) has painted pixels")

    for i, stroke in enumerate(strokes):
        pts = np.array(stroke["points"])
        check(pts.shape[1] == 3,
              f"Stroke {i} ('{stroke['structure']}'): points have 3 columns [x,y,width]")
        check(np.all(pts[:, 0] >= 0) and np.all(pts[:, 0] < W),
              f"Stroke {i}: all x coords within [0, {W})")
        check(np.all(pts[:, 1] >= 0) and np.all(pts[:, 1] < H),
              f"Stroke {i}: all y coords within [0, {H})")
        check(np.all(pts[:, 2] > 0),
              f"Stroke {i}: all width values > 0")

print(f"\n{'='*40}")
print(f"  Passed: {passed}   Failed: {failed}")
if failed == 0:
    print("  All tests passed ✅")
else:
    print("  Some tests FAILED ❌")


── 00_000001-1.2.840.4267.32.204129199654584020110347304149310718377.jpg ──


error: OpenCV(4.10.0) :-1: error: (-5:Bad argument) in function 'line'
> Overload resolution failed:
>  - Layout of the output array img is incompatible with cv::Mat
>  - Expected Ptr<cv::UMat> for argument 'img'
